# Milestone 1 — Single-GPU Qwen ES Reference

## 1. Setup + Config
## 2. Generation + Reward
## 3. Reward Standardization
## 4. Noise / Restore
## 5. Candidate Evaluation
## 6. Population
## 7. One Full ES Update
## 8. Save / Reload



In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers
import torch
import torch.nn as nn
import re


MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
load_dtype = torch.float16 if device == "cuda" else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=load_dtype,
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
)

model.eval()

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [2]:




GENERATION_CONFIG = {
    "system_prompt":
        "You are a calculator. Answer with only with number",

    "max_new_tokens": 16,
    "do_sample": False,

    "add_generation_prompt": True,
}


REFERENCE_CONFIG = {
    # Model/runtime
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "backend": "transformers",
    "dtype": str(next(model.parameters()).dtype),
    "device": str(next(model.parameters()).device),

    # ES
    "es_recipe": "one_point_gaussian_standardized",
    "sigma": 1e-3,
    "population_size": 4,
    "seeds": [0, 1, 2, 3],
    "learning_rate": 1e-4,

    # Numerical
    "reward_std_eta": 1e-8,

    # Generation
    "do_sample": False,

    # Reward
    "reward_type": "exact_integer_match",

    # Environment
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
}

for key, value in REFERENCE_CONFIG.items():
    print(f"{key:24s}: {value}")

model                   : Qwen/Qwen2.5-0.5B-Instruct
backend                 : transformers
dtype                   : torch.float16
device                  : cuda:0
es_recipe               : one_point_gaussian_standardized
sigma                   : 0.001
population_size         : 4
seeds                   : [0, 1, 2, 3]
learning_rate           : 0.0001
reward_std_eta          : 1e-08
do_sample               : False
reward_type             : exact_integer_match
torch_version           : 2.10.0+cu128
transformers_version    : 5.5.0


In [3]:
def standardize_rewards(rewards, eta=1e-8):
    # check each reward if it has NaN or Inf value.
    if not torch.isfinite(rewards).all():
        raise ValueError("Rewards contain NaN or Inf.")

    mean = rewards.mean()
    std = rewards.std(unbiased=False)

    if std < eta:
        z = torch.zeros_like(rewards)
    else:
        z = (rewards - mean) / (std + eta)

    return z, mean, std

In [4]:
# Case 1: normal population
r1 = torch.tensor([0.1875, 0.125, 0.25, 0.25])

z1, mean1, std1 = standardize_rewards(r1)

print("Normal case")
print("rewards:", r1)
print("mean   :", mean1.item())
print("std    :", std1.item())
print("z      :", z1)
print("z mean :", z1.mean().item())

Normal case
rewards: tensor([0.1875, 0.1250, 0.2500, 0.2500])
mean   : 0.203125
std    : 0.05182226374745369
z      : tensor([-0.3015, -1.5076,  0.9045,  0.9045])
z mean : 0.0


In [5]:
# Case 2: all rewards equal
r2 = torch.tensor([0.25, 0.25, 0.25, 0.25])

z2, mean2, std2 = standardize_rewards(r2)

print("Equal-reward case")
print("rewards:", r2)
print("mean   :", mean2.item())
print("std    :", std2.item())
print("z      :", z2)

Equal-reward case
rewards: tensor([0.2500, 0.2500, 0.2500, 0.2500])
mean   : 0.25
std    : 0.0
z      : tensor([0., 0., 0., 0.])


In [6]:
# Case 3: invalid reward
r3 = torch.tensor([0.25, float("nan"), 0.0, 0.25])

try:
    standardize_rewards(r3)
except ValueError as e:
    print("Correctly rejected:", e)

Correctly rejected: Rewards contain NaN or Inf.


# import from ES_Qwen


In [7]:
def generate_answer(model, tokenizer, question, max_new_tokens=16):
    device = next(model.parameters()).device
    messages = [
        {
            "role": "system",
            "content": "You are a calculator. Answer with only with number"
        },
        {
            "role": "user",
            "content": question
        },
    ]


    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt", # return pytorch format
        return_dict=True, # return dictionary of tensor (if it has attention mask or something)
        add_generation_prompt=True, # special token to tell model to start generate
        tokenize=True, # tokenize from string to int

    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )


    prompt_length = inputs["input_ids"].shape[1]

    generated_ids = output_ids[:,prompt_length:]

    answer = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True, # remove special token
    )[0]

    return answer.strip()



In [8]:
def extract_integer(text):
    numbers = re.findall(r"-?\d+", text.replace(",", ""))

    if not numbers:
        return None


    return int(numbers[-1])

In [9]:
def exact_reward(output_text, expected_answer):
    prediction = extract_integer(output_text)

    if prediction == expected_answer:
        reward = 1.0
    else:
        reward = 0.0

    return reward, prediction

In [10]:
examples = [
    # Easy
    {"question": "What is 17 + 24?", "answer": 41},
    {"question": "What is 6 * 8?", "answer": 48},
    {"question": "What is 91 - 37?", "answer": 54},
    {"question": "What is 12 * 13?", "answer": 156},

    # Medium
    {"question": "What is 137 * 46?", "answer": 6302},
    {"question": "What is 375 + 918 - 427?", "answer": 866},
    {"question": "What is 23 * 17 + 48?", "answer": 439},
    {"question": "What is (144 / 12) * 37?", "answer": 444},

    # Medium-hard
    {"question": "What is 218 * 37?", "answer": 8066},
    {"question": "What is 84 * 19 - 76?", "answer": 1520},
    {"question": "What is 312 * 27 - 145?", "answer": 8279},
    {"question": "What is (73 + 29) * 18?", "answer": 1836},

    # Word problems
    {
        "question":
        "A shop has 48 boxes with 36 pens in each box. "
        "It sells 725 pens. How many pens remain?",
        "answer": 1003,
    },
    {
        "question":
        "A school buys 36 packs of notebooks with 24 notebooks per pack. "
        "It distributes 579 notebooks. How many remain?",
        "answer": 285,
    },
    {
        "question":
        "A train travels 72 km per hour for 5 hours, "
        "then another 148 km. What is the total distance?",
        "answer": 508,
    },
    {
        "question":
        "A factory makes 125 items per hour for 18 hours. "
        "347 items are defective. How many usable items remain?",
        "answer": 1903,
    },
]

In [11]:
print("=== GENERATION / REWARD SMOKE TEST ===")

example = examples[0]

print("Question :", example["question"])
print("Expected :", example["answer"])

text = generate_answer(
    model,
    tokenizer,
    example["question"],
)

prediction = extract_integer(text)

reward = exact_reward(
    text,
    example["answer"],
)

print("Raw output :", repr(text))
print("Prediction :", prediction)
print("Reward     :", reward)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== GENERATION / REWARD SMOKE TEST ===
Question : What is 17 + 24?
Expected : 41
Raw output : '41'
Prediction : 41
Reward     : (1.0, 41)


In [12]:
def evaluate_model(model, tokenizer, examples):
    records = []
    total_reward = 0.0

    for example in examples:
        question = example["question"]
        expected = example["answer"]

        output_text = generate_answer(
            model,
            tokenizer,
            question,
        )

        reward, prediction = exact_reward(
            output_text,
            expected,
        )

        total_reward += reward

        records.append({
            "question": question,
            "expected": expected,
            "output_text": output_text,
            "prediction": prediction,
            "reward": reward,
        })

    mean_reward = total_reward / len(examples)

    return mean_reward, records

In [13]:
mean_reward, records = evaluate_model(
    model,
    tokenizer,
    examples,
)

print("Mean reward:", mean_reward)

for record in records:
    print(
        record["question"],
        "| expected:", record["expected"],
        "| prediction:", record["prediction"],
        "| reward:", record["reward"],
    )

Mean reward: 0.25
What is 17 + 24? | expected: 41 | prediction: 41 | reward: 1.0
What is 6 * 8? | expected: 48 | prediction: 48 | reward: 1.0
What is 91 - 37? | expected: 54 | prediction: 54 | reward: 1.0
What is 12 * 13? | expected: 156 | prediction: 156 | reward: 1.0
What is 137 * 46? | expected: 6302 | prediction: 6292 | reward: 0.0
What is 375 + 918 - 427? | expected: 866 | prediction: 664 | reward: 0.0
What is 23 * 17 + 48? | expected: 439 | prediction: 509 | reward: 0.0
What is (144 / 12) * 37? | expected: 444 | prediction: 589 | reward: 0.0
What is 218 * 37? | expected: 8066 | prediction: 7906 | reward: 0.0
What is 84 * 19 - 76? | expected: 1520 | prediction: 1302 | reward: 0.0
What is 312 * 27 - 145? | expected: 8279 | prediction: 8095 | reward: 0.0
What is (73 + 29) * 18? | expected: 1836 | prediction: 10 | reward: 0.0
A shop has 48 boxes with 36 pens in each box. It sells 725 pens. How many pens remain? | expected: 1003 | prediction: None | reward: 0.0
A school buys 36 packs 

In [14]:
# full model perturb

@torch.no_grad()
def apply_perturbation(model, seed, sigma):
    generator = torch.Generator(
        device = next(model.parameters()).device
    )
    generator.manual_seed(seed)

    for param in model.parameters():
        if not param.is_floating_point():
            continue


        epsilon = torch.randn_like(
            param,
            generator=generator
        )

        param.add_(epsilon, alpha=sigma)



@torch.no_grad()
def revert_perturbation(model, seed, sigma):
    generator = torch.Generator(
        device = next(model.parameters()).device
    )
    generator.manual_seed(seed)

    for param in model.parameters():
        if not param.is_floating_point():
            continue


        epsilon = torch.randn_like(
            param,
            generator=generator
        )

        param.sub_(epsilon, alpha=sigma)






In [15]:
# take snapshot of a model
@torch.no_grad()
def take_snapshot(model, max_param=8, value_per_param=256,):
    snapshot = {}

    for name, param in model.named_parameters():
        if not param.is_floating_point():
            continue

        snapshot[name] = (
            param.detach().flatten()[:value_per_param].cpu().clone()
        )

        if len(snapshot) >= max_param:
            break

    return snapshot


In [16]:
def snapshot_max_diff(snapshot_a, snapshot_b):
    max_diff = 0.0

    for name in snapshot_a:
        diff = (
            snapshot_a[name] - snapshot_b[name]
        ).abs().max().item()

        max_diff = max(max_diff, diff)

    return max_diff

In [17]:
print("=== PERTURB / RESTORE SMOKE TEST ===")

seed = 0
sigma = REFERENCE_CONFIG["sigma"]

# 1. canonical state
snapshot_before = take_snapshot(model)

# 2. perturb
apply_perturbation(
    model,
    seed=seed,
    sigma=sigma,
)

snapshot_perturbed = take_snapshot(model)

perturb_diff = snapshot_max_diff(
    snapshot_before,
    snapshot_perturbed,
)

# 3. restore
revert_perturbation(
    model,
    seed=seed,
    sigma=sigma,
)

snapshot_after = take_snapshot(model)

restore_diff = snapshot_max_diff(
    snapshot_before,
    snapshot_after,
)

print("Seed         :", seed)
print("Sigma        :", sigma)
print("Perturb diff :", perturb_diff)
print("Restore diff :", restore_diff)

=== PERTURB / RESTORE SMOKE TEST ===
Seed         : 0
Sigma        : 0.001
Perturb diff : 0.00390625
Restore diff : 0.001953125


In [18]:
def evaluate_candidate(
    model,
    tokenizer,
    examples,
    seed,
    sigma,
):
    snapshot_before = take_snapshot(model)

    apply_perturbation(
        model,
        seed=seed,
        sigma=sigma,
    )


    # to avoid generate being ERROR like OOM, exception ..., it still revert model
    try:
        reward, records = evaluate_model(
            model,
            tokenizer,
            examples,
        )

    finally:
        revert_perturbation(
            model,
            seed=seed,
            sigma=sigma,
        )

    snapshot_after = take_snapshot(model)

    restore_diff = snapshot_max_diff(
        snapshot_before,
        snapshot_after,
    )

    return {
        "seed": seed,
        "sigma": sigma,
        "reward": reward,
        "records": records,
        "restore_diff": restore_diff,
    }

In [19]:
print("=== ONE CANDIDATE SMOKE TEST ===")

candidate = evaluate_candidate(
    model=model,
    tokenizer=tokenizer,
    examples=examples,
    seed=0,
    sigma=REFERENCE_CONFIG["sigma"],
)

print("Seed         :", candidate["seed"])
print("Sigma        :", candidate["sigma"])
print("Reward       :", candidate["reward"])
print("Restore diff :", candidate["restore_diff"])

=== ONE CANDIDATE SMOKE TEST ===


Seed         : 0
Sigma        : 0.001
Reward       : 0.25
Restore diff : 0.0


In [20]:
def evaluate_population(
    model,
    tokenizer,
    examples,
    seeds,
    sigma,
):
    results = []

    for seed in seeds:
        result = evaluate_candidate(
            model=model,
            tokenizer=tokenizer,
            examples=examples,
            seed=seed,
            sigma=sigma,
        )

        results.append(result)

    return results

In [21]:
print("=== POPULATION TEST ===")

seeds = REFERENCE_CONFIG["seeds"]
sigma = REFERENCE_CONFIG["sigma"]

results = evaluate_population(
    model=model,
    tokenizer=tokenizer,
    examples=examples,
    seeds=seeds,
    sigma=sigma,
)

for result in results:
    print(
        "seed:",
        result["seed"],
        "| reward:",
        result["reward"],
        "| restore_diff:",
        result["restore_diff"],
    )

=== POPULATION TEST ===
seed: 0 | reward: 0.25 | restore_diff: 0.0
seed: 1 | reward: 0.1875 | restore_diff: 1.52587890625e-05
seed: 2 | reward: 0.25 | restore_diff: 0.0009765625
seed: 3 | reward: 0.25 | restore_diff: 6.103515625e-05


In [22]:
rewards = torch.tensor(
    [result["reward"] for result in results],
    dtype=torch.float32,
)

z, reward_mean, reward_std = standardize_rewards(
    rewards,
    eta=REFERENCE_CONFIG["reward_std_eta"],
)

print("Rewards :", rewards)
print("Mean    :", reward_mean.item())
print("Std     :", reward_std.item())
print("Z       :", z)

print("Std > 0       :", reward_std.item() > 0)
print("Z all zero    :", torch.all(z == 0).item())

Rewards : tensor([0.2500, 0.1875, 0.2500, 0.2500])
Mean    : 0.234375
Std     : 0.027063293382525444
Z       : tensor([ 0.5774, -1.7321,  0.5774,  0.5774])
Std > 0       : True
Z all zero    : False


In [23]:
@torch.no_grad()
def es_update_direction(
    model,
    seeds,
    z_scores,
    learning_rate,
):
    population_size = len(seeds)

    # One RNG stream for each candidate.
    generators = []

    device = next(model.parameters()).device

    for seed in seeds:
        g = torch.Generator(device=device)
        g.manual_seed(seed)
        generators.append(g)

    # Build and apply the ES direction parameter by parameter.
    for param in model.parameters():

        if not param.is_floating_point():
            continue

        direction = torch.zeros_like(param)

        for i, g in enumerate(generators):
            epsilon = torch.randn(
                param.shape,
                generator=g,
                device=param.device,
                dtype=param.dtype,
            )

            direction.add_(
                epsilon,
                # because alpha want a python float, not tensor float
                alpha=float(z_scores[i]),
            )

        direction.div_(population_size)

        param.add_(
            direction,
            alpha=learning_rate,
        )

In [24]:
print("=== BEFORE ES UPDATE ===")

reward_before, _ = evaluate_model(
    model,
    tokenizer,
    examples,
)

snapshot_before_update = take_snapshot(model)

print("Base reward:", reward_before)

=== BEFORE ES UPDATE ===
Base reward: 0.25


In [25]:
print("=== APPLY PERMANENT ES UPDATE ===")

learning_rate = REFERENCE_CONFIG["learning_rate"]

es_update_direction(
    model=model,
    seeds=REFERENCE_CONFIG["seeds"],
    z_scores=z,
    learning_rate=learning_rate,
)

print("Update applied.")

=== APPLY PERMANENT ES UPDATE ===
Update applied.


In [26]:
snapshot_after_update = take_snapshot(model)

update_diff = snapshot_max_diff(
    snapshot_before_update,
    snapshot_after_update,
)

print("Update max diff:", update_diff)

Update max diff: 0.00018310546875


In [27]:
print("=== AFTER ES UPDATE ===")

reward_after, records_after = evaluate_model(
    model,
    tokenizer,
    examples,
)

print("Reward before:", reward_before)
print("Reward after :", reward_after)
print("Update diff  :", update_diff)

=== AFTER ES UPDATE ===
Reward before: 0.25
Reward after : 0.25
Update diff  : 0.00018310546875


In [28]:
# save
CHECKPOINT_DIR = "./qwen_es_milestone1_gen1"

model.save_pretrained(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)

print("save to: ", CHECKPOINT_DIR)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

save to:  ./qwen_es_milestone1_gen1


In [29]:
# load local model like load hugging face model

reloaded_tokenizer = AutoTokenizer.from_pretrained(
    CHECKPOINT_DIR,
)

reloaded_model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT_DIR,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
).to(device)

reloaded_model.eval()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [30]:
print("=== RELOADED MODEL EVAL ===")

reload_reward, reload_records = evaluate_model(
    reloaded_model,
    reloaded_tokenizer,
    examples,
)

print("Reward after update :", reward_after)
print("Reward after reload :", reload_reward)

=== RELOADED MODEL EVAL ===
Reward after update : 0.25
Reward after reload : 0.25


In [31]:
question = "What is 17 + 24?"

answer_current = generate_answer(
    model,
    tokenizer,
    question,
)

answer_reloaded = generate_answer(
    reloaded_model,
    reloaded_tokenizer,
    question,
)

print("Current model :", repr(answer_current))
print("Reloaded model:", repr(answer_reloaded))

Current model : '41'
Reloaded model: '41'


In [32]:
del reloaded_model
del reloaded_tokenizer

import gc
gc.collect()

torch.cuda.empty_cache()

# Milestone 2 — Numerical Contract + Multi-Generation

## 9. Parameter Schema
## 10. Tied / Shared Parameters
## 11. Canonical Restore
## 12. Noise Reconstruction Consistency
## 13. One-Generation Regression Test
## 14. Multi-Generation Loop

In [33]:
import json
import hashlib


def build_parameter_schema(model):
    schema = [] # list of dictionary (store info of each param )

    for index, (name, param) in enumerate(model.named_parameters()):
        if not param.is_floating_point():
            raise TypeError(
                f"Unexpected non-floating parameter: "
                f"{name} ({param.dtype})"
            )

        schema.append(
            {
                "index": index,
                "name": name,
                "shape": list(param.shape),
                "dtype": str(param.dtype),
                "numel": param.numel(),
            }
        )

    return schema





In [34]:
def hash_parameter_schema(schema):
    # convert to json type
    serialized = json.dumps(
        schema, # list of dict
        sort_keys=True, # sort dict by keys to keep consistency
        separators=(",", ":"), # remove white space
    )

    # hashing
    schema_hash = hashlib.sha256( # hashing byte to 256 bit hashed
        serialized.encode("utf-8") # convert str -> byte using utf8 format
    ).hexdigest()  # convert 256 bit to hex

    return schema_hash



In [35]:
parameter_schema = build_parameter_schema(model)

parameter_schema_hash = hash_parameter_schema(
    parameter_schema
)

print("Number of parameter tensors:",
      len(parameter_schema))

print(
    "Total parameter elements:",
    sum(item["numel"] for item in parameter_schema)
)

print(
    "Parameter schema SHA256:",
    parameter_schema_hash
)

Number of parameter tensors: 290
Total parameter elements: 494032768
Parameter schema SHA256: 152e9d82e61d6610a414466026ac60a13e718924dfdf0416809686adcf4188e1


In [36]:
parameter_schema_2 = build_parameter_schema(model)

parameter_schema_hash_2 = hash_parameter_schema(
    parameter_schema_2
)

print("Hash 1:", parameter_schema_hash)
print("Hash 2:", parameter_schema_hash_2)

print(
    "Same schema:",
    parameter_schema_hash
    == parameter_schema_hash_2
)

Hash 1: 152e9d82e61d6610a414466026ac60a13e718924dfdf0416809686adcf4188e1
Hash 2: 152e9d82e61d6610a414466026ac60a13e718924dfdf0416809686adcf4188e1
Same schema: True


In [37]:
from collections import defaultdict
# default dict will create a default type which you explicitly define it


# find named of param which share with other
def find_parameter_aliases(model):
    aliases = defaultdict(list) # dict (default is list)

    # create dict of aliases, key is id of object (usually mem addr)
    for name, param in model.named_parameters(remove_duplicate=False):
        # each param has unique id, and value of each id is name
        aliases[id(param)].append(name)

    # count how many id have duplicate name
    # create list of names being alias to 1 param
    aliases_group = [
        names for names in aliases.values() if len(names) > 2
    ]

    return aliases_group







In [38]:
alias_groups = find_parameter_aliases(model)

print(
    "Number of shared/tied parameter groups:",
    len(alias_groups)
)

for group in alias_groups:
    print(group)

Number of shared/tied parameter groups: 0


In [39]:
# capture all paramters of a model

def capture_canonical_parameters(model):
    snapshot = {}

    for name, param in model.named_parameters():
        snapshot[name] = (
            param.detach() # detach (take numerical only, ignore autograd)
            .cpu() # move to cpu, vram -> ram
            .clone()
        )

    return snapshot


# restore canonical param
@torch.no_grad()
def restore_canonical_parameters(model, snapshot):
    for name, param in model.named_parameters():
        # check name of model exist in snapshot
        if name not in snapshot:
            raise KeyError(
                f"Missing parameter in canonical snapshot: {name}"
            )

        # check param shape of each name
        if tuple(snapshot[name].shape) != tuple(param.shape):
            raise ValueError(
                f"Shape mismatch for {name}: "
                f"snapshot={tuple(source.shape)}, "
                f"model={tuple(param.shape)}"
            )


        # copy
        param.copy_(
            snapshot[name].to(param.device, dtype=param.dtype)
        )


In [40]:
# compare full param diff
@torch.no_grad()
def full_parameter_max_diff(model, snapshot):
    max_diff = 0.0
    worst_parameter_diff = None

    for name, param in model.named_parameters():
        reference_param = snapshot[name].to(param.device, param.dtype)

        diff = (reference_param - param).abs().max().item()

        if diff >= max_diff:
            max_diff = diff
            worst_parameter_diff = name

        if max_diff == 0.0:
            worst_parameter_diff = None

    return max_diff, worst_parameter_diff


In [41]:
print("=== CAPTURE CANONICAL MODEL ===")

canonical_snapshot = capture_canonical_parameters(
    model
)

print(
    "Captured parameter tensors:",
    len(canonical_snapshot)
)

=== CAPTURE CANONICAL MODEL ===


Captured parameter tensors: 290


In [42]:
print("=== CANONICAL RESTORE TEST ===")

seed = 0
sigma = REFERENCE_CONFIG["sigma"]

apply_perturbation(
    model,
    seed=seed,
    sigma=sigma,
)
perturb_diff, perturb_worst = (
    full_parameter_max_diff(
        model,
        canonical_snapshot,
    )
)

print("After perturb:")
print("Max diff :", perturb_diff)
print("Worst    :", perturb_worst)

=== CANONICAL RESTORE TEST ===
After perturb:
Max diff : 0.00634765625
Worst    : model.layers.21.mlp.up_proj.weight


In [43]:
restore_canonical_parameters(
    model,
    canonical_snapshot,
)

restore_diff, restore_worst = (
    full_parameter_max_diff(
        model,
        canonical_snapshot,
    )
)

print("\nAfter canonical restore:")
print("Max diff :", restore_diff)
print("Worst    :", restore_worst)


After canonical restore:
Max diff : 0.0
Worst    : None


In [44]:
@torch.no_grad()
def make_noise_generator(model, seed):
    device = next(model.parameters()).device
    g = torch.Generator(
        device=device
    )
    g.manual_seed(seed)

    return g


In [45]:
@torch.no_grad()
def reconstruct_specific_parameter_noise(model, seed, target_name):
    generator = make_noise_generator(model, seed)

    # loop to target name, if not found: raise error

    for name, param in model.named_parameters():
        if not param.is_floating_point():
            raise TypeError(
                f"Unexpected non-floating parameter: "
                f"{name} ({param.dtype})"
            )

        epsilon = torch.randn_like(
            param,
            generator=generator,
        )

        if name == target_name:
            return epsilon

    raise KeyError(
        f"Parameter not found: {target_name}"
    )

In [46]:
target_name = (
    "model.layers.0.self_attn.q_proj.weight"
)

epsilon_1 = reconstruct_specific_parameter_noise(
    model,
    seed=0,
    target_name=target_name,
)

epsilon_2 = reconstruct_specific_parameter_noise(
    model,
    seed=0,
    target_name=target_name,
)

print("Target:", target_name)

print(
    "Exactly equal:",
    torch.equal(epsilon_1, epsilon_2)
)

print(
    "Max difference:",
    (epsilon_1 - epsilon_2)
    .abs()
    .max()
    .item()
)

print(
    "Noise shape:",
    tuple(epsilon_1.shape)
)

Target: model.layers.0.self_attn.q_proj.weight
Exactly equal: True
Max difference: 0.0
Noise shape: (896, 896)


In [47]:
epsilon_other_seed = reconstruct_specific_parameter_noise(
    model,
    seed=1,
    target_name=target_name,
)

print(
    "Seed 0 vs seed 1 equal:",
    torch.equal(
        epsilon_1,
        epsilon_other_seed,
    )
)

print(
    "Seed 0 vs seed 1 max diff:",
    (
        epsilon_1
        - epsilon_other_seed
    )
    .abs()
    .max()
    .item()
)

Seed 0 vs seed 1 equal: False
Seed 0 vs seed 1 max diff: 6.9921875


In [48]:
def iter_parameter_noise(model, seed):
    generator = make_noise_generator(
        model,
        seed,
    )

    for name, param in model.named_parameters():

        if not param.is_floating_point():
            raise TypeError(
                f"Unexpected non-floating parameter: "
                f"{name} ({param.dtype})"
            )

        epsilon = torch.randn_like(
            param,
            generator=generator,
        )
        # yield does not end function, so generator still alive
        yield name, param, epsilon

# Refactor Note — Toward a Canonical Noise Engine

The current notebook implementation duplicates random-noise generation logic across several functions such as perturbation, reconstruction, revert, and ES update.

For the notebook prototype, this is acceptable for learning and validation. However, for the real heterogeneous implementation, duplicated RNG logic is risky because any mismatch in seed handling, parameter traversal, dtype, or sampling procedure can cause the evaluated perturbation to differ from the perturbation reconstructed during update.

The implementation will therefore be refactored toward a single canonical noise-generation component.

Core idea:

- One source of truth for:
  - seed handling
  - PRNG creation
  - parameter traversal order
  - parameter schema validation
  - Gaussian sampling
  - dtype/device handling
- Candidate perturbation, replay, debugging, and ES update must all consume the same noise engine.
- The parameter schema hash is used to verify that the runtime is reconstructing noise over the expected model structure.
- Canonical model restoration should be separated from perturbation logic.

Prototype abstraction:

```text
Canonical Noise Engine
        |
        +--> perturb candidate
        |
        +--> reconstruct parameter noise
        |
        +--> ES update
        |
        +--> replay / consistency checks

In [49]:
def evaluate_candidate_v2(
    model,
    tokenizer,
    examples,
    seed,
    sigma,
    canonical_snapshot,
):
    apply_perturbation(
        model,
        seed=seed,
        sigma=sigma,
    )

    try:
        reward, records = evaluate_model(
            model,
            tokenizer,
            examples,
        )

    finally:
        restore_canonical_parameters(
            model,
            canonical_snapshot,
        )

    restore_diff, restore_worst = (
        full_parameter_max_diff(
            model,
            canonical_snapshot,
        )
    )

    return {
        "seed": seed,
        "sigma": sigma,
        "reward": reward,
        "records": records,
        "restore_diff": restore_diff,
        "restore_worst": restore_worst,
    }

In [50]:
restore_canonical_parameters(
    model,
    canonical_snapshot,
)

In [51]:
candidate_v2 = evaluate_candidate_v2(
    model=model,
    tokenizer=tokenizer,
    examples=examples,
    seed=0,
    sigma=REFERENCE_CONFIG["sigma"],
    canonical_snapshot=canonical_snapshot,
)

print("Reward       :", candidate_v2["reward"])
print("Restore diff :", candidate_v2["restore_diff"])
print("Restore worst:", candidate_v2["restore_worst"])

Reward       : 0.25
Restore diff : 0.0
Restore worst: None


In [52]:
def evaluate_population_v2(
    model,
    tokenizer,
    examples,
    seeds,
    sigma,
    canonical_snapshot,
):
    results = []

    for seed in seeds:
        result = evaluate_candidate_v2(
            model=model,
            tokenizer=tokenizer,
            examples=examples,
            seed=seed,
            sigma=sigma,
            canonical_snapshot=canonical_snapshot,
        )

        results.append(result)

    return results

In [53]:
results_v2 = evaluate_population_v2(
    model=model,
    tokenizer=tokenizer,
    examples=examples,
    seeds=REFERENCE_CONFIG["seeds"],
    sigma=REFERENCE_CONFIG["sigma"],
    canonical_snapshot=canonical_snapshot,
)

for result in results_v2:
    print(
        "seed:", result["seed"],
        "| reward:", result["reward"],
        "| restore:", result["restore_diff"],
    )

seed: 0 | reward: 0.25 | restore: 0.0
seed: 1 | reward: 0.1875 | restore: 0.0
seed: 2 | reward: 0.25 | restore: 0.0
seed: 3 | reward: 0.25 | restore: 0.0


In [54]:
rewards_v2 = torch.tensor(
    [result["reward"] for result in results_v2],
    dtype=torch.float32,
)

z_v2, reward_mean_v2, reward_std_v2 = (
    standardize_rewards(
        rewards_v2,
        eta=REFERENCE_CONFIG["reward_std_eta"],
    )
)

print("Rewards :", rewards_v2)
print("Mean    :", reward_mean_v2.item())
print("Std     :", reward_std_v2.item())
print("Z       :", z_v2)

Rewards : tensor([0.2500, 0.1875, 0.2500, 0.2500])
Mean    : 0.234375
Std     : 0.027063293382525444
Z       : tensor([ 0.5774, -1.7321,  0.5774,  0.5774])


In [55]:
canonical_diff, canonical_worst = (
    full_parameter_max_diff(
        model,
        canonical_snapshot,
    )
)

print("Before update canonical diff :", canonical_diff)
print("Worst parameter              :", canonical_worst)

Before update canonical diff : 0.0
Worst parameter              : None


In [56]:
reward_before_v2, _ = evaluate_model(
    model,
    tokenizer,
    examples,
)

print("Reward before update:", reward_before_v2)

Reward before update: 0.25


In [57]:
es_update_direction(
    model=model,
    seeds=REFERENCE_CONFIG["seeds"],
    z_scores=z_v2,
    learning_rate=REFERENCE_CONFIG["learning_rate"],
)

In [58]:
update_diff_v2, update_worst_v2 = (
    full_parameter_max_diff(
        model,
        canonical_snapshot,
    )
)

print("Update max diff :", update_diff_v2)
print("Update worst    :", update_worst_v2)

Update max diff : 0.00030517578125
Update worst    : model.embed_tokens.weight


In [59]:
reward_after_v2, _ = evaluate_model(
    model,
    tokenizer,
    examples,
)

print("Reward before :", reward_before_v2)
print("Reward after  :", reward_after_v2)
print("Update diff   :", update_diff_v2)

Reward before : 0.25
Reward after  : 0.25
Update diff   : 0.00030517578125


In [60]:
def run_es_generation(
    model,
    tokenizer,
    examples,
    seeds,
    sigma,
    learning_rate,
    reward_std_eta,
    generation_id,
):
    # capture canonical model
    canonical_snapshot = capture_canonical_parameters(model)

    # base reward
    reward_before, _ = evaluate_model(model, tokenizer, examples)

    # population
    results = evaluate_population_v2(
        model=model,
        tokenizer=tokenizer,
        examples=examples,
        seeds=seeds,
        sigma=sigma,
        canonical_snapshot=canonical_snapshot,
    )

    # rewards + z rewards
    rewards = torch.tensor(
        [result["reward"] for result in results],
        dtype=torch.float32,
    )

    z, reward_mean, reward_std = standardize_rewards(
        rewards,
        eta=reward_std_eta,
    )



    # verify canonical state
    canonical_diff, canonical_worst = full_parameter_max_diff(model, canonical_snapshot)


    if canonical_diff != 0.0:
        raise RuntimeError(
            f"Canonical model was not restored. "
            f"max_diff={canonical_diff}, "
            f"worst={canonical_worst}"
        )

    # permanent update
    # has signal check if any z != 0
    has_signal = torch.any(z != 0).item()
    if has_signal:
        es_update_direction(
            model=model,
            seeds=seeds,
            z_scores=z,
            learning_rate=learning_rate,
        )


    # update diff
    update_diff, update_worst = full_parameter_max_diff(model, canonical_snapshot)

    # reward after
    reward_after, _ = evaluate_model(model, tokenizer, examples)

    # log
    log = {
        "generation": generation_id,

        "reward_before": reward_before,

        "candidate_rewards": rewards.tolist(),
        "reward_mean": reward_mean.item(),
        "reward_std": reward_std.item(),
        "z_scores": z.tolist(),

        "has_signal": has_signal,

        "update_max_diff": update_diff,
        "update_worst_parameter": update_worst,

        "reward_after": reward_after,

        "candidate_restore_diffs": [
            result["restore_diff"]
            for result in results
        ],
    }

    return log

In [61]:
generation_log = run_es_generation(
    model=model,
    tokenizer=tokenizer,
    examples=examples,

    seeds=REFERENCE_CONFIG["seeds"],
    sigma=REFERENCE_CONFIG["sigma"],
    learning_rate=REFERENCE_CONFIG["learning_rate"],
    reward_std_eta=REFERENCE_CONFIG["reward_std_eta"],

    generation_id=0,
)

In [62]:
print("\n=== GENERATION SUMMARY ===")

for key, value in generation_log.items():
    print(f"{key}: {value}")


=== GENERATION SUMMARY ===
generation: 0
reward_before: 0.25
candidate_rewards: [0.25, 0.1875, 0.25, 0.25]
reward_mean: 0.234375
reward_std: 0.027063293382525444
z_scores: [0.5773500800132751, -1.7320502996444702, 0.5773500800132751, 0.5773500800132751]
has_signal: True
update_max_diff: 0.00030517578125
update_worst_parameter: model.embed_tokens.weight
reward_after: 0.25
candidate_restore_diffs: [0.0, 0.0, 0.0, 0.0]


In [63]:
history = []

num_generations = 3

for generation in range(num_generations):
    log = run_es_generation(
        model=model,
        tokenizer=tokenizer,
        examples=examples,

        seeds=REFERENCE_CONFIG["seeds"],
        sigma=REFERENCE_CONFIG["sigma"],
        learning_rate=REFERENCE_CONFIG["learning_rate"],
        reward_std_eta=REFERENCE_CONFIG["reward_std_eta"],

        generation_id=generation,
    )

    history.append(log)

    print(
        f"gen={generation} "
        f"| before={log['reward_before']:.4f} "
        f"| mean_candidate={log['reward_mean']:.4f} "
        f"| std={log['reward_std']:.4f} "
        f"| after={log['reward_after']:.4f} "
        f"| update_diff={log['update_max_diff']:.6g}"
    )

gen=0 | before=0.2500 | mean_candidate=0.2344 | std=0.0271 | after=0.2500 | update_diff=0.000305176
gen=1 | before=0.2500 | mean_candidate=0.2344 | std=0.0271 | after=0.2500 | update_diff=0.000305176
gen=2 | before=0.2500 | mean_candidate=0.2344 | std=0.0271 | after=0.2500 | update_diff=0.000305176


In [64]:
print("\n=== TRAINING HISTORY ===")

for log in history:
    print(
        log["generation"],
        "| before:", log["reward_before"],
        "| candidates:", log["candidate_rewards"],
        "| std:", log["reward_std"],
        "| signal:", log["has_signal"],
        "| update:", log["update_max_diff"],
        "| after:", log["reward_after"],
    )


=== TRAINING HISTORY ===
0 | before: 0.25 | candidates: [0.25, 0.1875, 0.25, 0.25] | std: 0.027063293382525444 | signal: True | update: 0.00030517578125 | after: 0.25
1 | before: 0.25 | candidates: [0.25, 0.1875, 0.25, 0.25] | std: 0.027063293382525444 | signal: True | update: 0.00030517578125 | after: 0.25
2 | before: 0.25 | candidates: [0.25, 0.1875, 0.25, 0.25] | std: 0.027063293382525444 | signal: True | update: 0.00030517578125 | after: 0.25


In [65]:
all_restore_clean = all(
    diff == 0.0
    for log in history
    for diff in log["candidate_restore_diffs"]
)

print(
    "All candidate restores exact:",
    all_restore_clean
)

All candidate restores exact: True


In [66]:
import json

history_serializable = history

with open(
    "./milestone2_history.json",
    "w",
) as f:
    json.dump(
        history_serializable,
        f,
        indent=2,
    )

print("Saved milestone2_history.json")

Saved milestone2_history.json


In [67]:
FINAL_CHECKPOINT_DIR = (
    "./qwen_es_milestone2_final"
)

model.save_pretrained(
    FINAL_CHECKPOINT_DIR
)

tokenizer.save_pretrained(
    FINAL_CHECKPOINT_DIR
)

print(
    "Saved final checkpoint:",
    FINAL_CHECKPOINT_DIR
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved final checkpoint: ./qwen_es_milestone2_final
